# 第二部分：线性代数 —— AI 的数据语言

## 第3章　标量、向量与张量 —— AI的数据容器

> 本章目标：建立"数据形状（shape）= 张量的维度结构"这条核心直觉。能从 shape 一眼判断是标量、向量、矩阵还是高阶张量，理解轴（axis）的含义——`axis=0` 是"按行"还是"按列"不再是玄学，而是可以用 `np.sum` 亲手验证的物理事实。
> 前置知识：第 1 章（NumPy 数组、`np.linspace`、函数）



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
print("env ready")



### 3.1　标量（Scalar）—— 0 维数字，AI 世界的"原子"

训练神经网络时，你最关心的那个数字——"loss = 2.37"——就是一个标量。它没有方向、没有长度、没有"第几个"的概念，它就是一个孤零零的数。

📐 **定义　标量（Scalar）**：0 维张量，shape = `()`，只包含一个数值。在 Python 中是普通的 `float` 或 `int`，在 NumPy/PyTorch 中是 `np.array(3.14)` 或 `torch.tensor(2.37)`。

标量虽然简单，但在深度学习中承担着最重要的角色：**损失函数的值。** 整个训练过程的唯一目标就是让这个数变小。反向传播时，所有参数的梯度都是从这一个标量出发往回算的——它是一切的终点，也是一切的起点。

💻 **代码　亲手触摸标量：Python 数字 vs NumPy 0 维数组**




In [ ]:
import numpy as np

# Python 原生标量
loss_py = 2.37
print(f"Python float: 值={loss_py}, type={type(loss_py).__name__}")

# NumPy 0 维数组——本质上还是标量，但变成了 ndarray 对象
loss_np = np.array(2.37)
print(f"NumPy 标量: 值={loss_np}, type={type(loss_np).__name__}, "
      f"ndim={loss_np.ndim}, shape={loss_np.shape}")

# 关键操作：从 NumPy 标量取出 Python 数字
print(f"loss_np.item() = {loss_np.item()} → 变回 Python float")

# 标量运算：和普通数字一样使用
lr = 0.01
gradient = np.array(0.5)
new_loss = loss_np - lr * gradient  # 标量算术——逐元素运算对 0 维无意义
print(f"\n参数更新: loss - lr×grad = {loss_np} - {lr}×{gradient} = {new_loss}")
print(f"new_loss 的 shape: {new_loss.shape} ← 还是标量")




> **关键洞察**：`loss.item()` 是 PyTorch 训练循环中最常见的操作之一——它把 GPU 上的 0 维张量取出来变成一个 Python float，让你可以打印、记录、画图。在 `print(f"Epoch {epoch}: loss = {loss.item():.4f}")` 这行每个 AI 工程师都写过的代码背后，就是一个标量在 GPU 和 CPU 之间的一次"越狱"。

🔗 **AI 连接**：PyTorch 的 `loss.backward()` 就是从标量 loss 出发，沿着计算图反向传播梯度到每一个参数（第 14 章）。一个标量驱动了数以亿计的参数更新——这就是标量在深度学习中的真正分量。

---

### 3.2　向量（Vector）—— 1 维数组，有方向的"列表"

如果你要描述一个用户的画像，你不会只用"年龄=28"这一个数字——太单薄了。你会用一个列表：`[年龄=28, 收入=15000, 活跃度=0.7, 消费次数=42]`。这个列表的数学名字叫**向量**。

📐 **定义　向量（Vector）**：1 维张量，shape = `(n,)`，包含 n 个按顺序排列的数值。在 NumPy 中 `np.array([28, 15000, 0.7, 42])` 的 shape 是 `(4,)`。向量可以看作"从一个点到另一个点的箭头"——既有长度（模），又有方向。

向量在 AI 中无处不在——**每一个 token 嵌入、每一层隐藏状态、每一个偏置项，本质上都是向量。** 理解向量的 shape 是读懂 PyTorch 代码的基本功：看到 `x.shape = (768,)` 你就知道这是一个 768 维向量，看到 `bias.shape = (256,)` 你就知道这是一个 256 维的偏置向量。

💻 **代码　向量的创建、索引和基本属性**




In [ ]:
import numpy as np

# ===== 创建向量 =====
user_features = np.array([28, 15000, 0.7, 42])     # 用户特征向量
word_embedding = np.random.randn(768)                # 模拟 GPT-2 的词嵌入 (768维)
bias = np.zeros(256)                                 # 全零偏置向量

print(f"用户特征: {user_features}")
print(f"  shape={user_features.shape}, ndim={user_features.ndim}, dtype={user_features.dtype}")
print(f"词嵌入: shape={word_embedding.shape}, 前5维={word_embedding[:5].round(3)}")
print(f"偏置向量: shape={bias.shape}, 全零验证={bias.sum() == 0}")

# ===== 向量索引和切片——和 Python 列表一样直观 =====
v = np.array([10, 20, 30, 40, 50])
print(f"\n向量 v = {v}")
print(f"v[0] = {v[0]}   (第一个元素——索引从 0 开始)")
print(f"v[-1] = {v[-1]}  (最后一个元素)")
print(f"v[1:4] = {v[1:4]} (第 2 到第 4 个元素)")
print(f"v[:3] = {v[:3]}   (前 3 个)")

# ===== 向量的"长度"——L2 范数 =====
norm = np.sqrt(np.sum(user_features.astype(float) ** 2))
print(f"\n用户特征向量的 L2 范数: {norm:.1f}")
# L2 范数描述向量"从原点到终点的直线距离"——是相似度计算的基础（第 5 章）




> **关键洞察**：向量 shape 的表示是 `(n,)` 而不是 `(n, 1)` 或 `(1, n)`——这个逗号很重要。`(768,)` 是一维向量，`(768, 1)` 是二维列矩阵，`(1, 768)` 是二维行矩阵。三者在数学上可以表示同样的 768 个数字，但在 NumPy/PyTorch 的广播机制中行为完全不同（第 6.5 节详解）。

🔗 **AI 连接**：第 4 章用向量加减演示词嵌入的语义运算（国王−男人+女人≈女王）；第 5 章用向量点积计算两个嵌入的相似度（Transformer 中 Q·K 就是两个向量的点积，第 29 章）；第 12 章的梯度 ∇L 本身就是一个向量——每一个分量是损失对一个参数的偏导数。

---

### 3.3　矩阵（Matrix）—— 2 维表格，向量的"批量打包"

一句话概括矩阵：**把多个向量叠在一起。** 如果你有 32 张图片，每张图片用 784 个像素值描述，与其用 32 个单独的向量，不如把它们摞成一个 32×784 的矩阵——每一行是一个样本，每一列是一个特征。这就是 AI 数据集的标配格式。

📐 **定义　矩阵（Matrix）**：2 维张量，shape = `(m, n)`，m 行 n 列。在深度学习中，**第一维几乎总是 batch（样本数），第二维是特征数。** `X.shape = (32, 784)` 意味着 32 个样本，每个 784 维。

💻 **代码　矩阵的创建与形状直觉**




In [ ]:
import numpy as np

# ===== 模拟真实场景 =====
batch_size, num_features = 32, 784
X = np.random.randn(batch_size, num_features)  # 32 个样本，每个 784 维

print(f"批量输入 X:")
print(f"  shape = {X.shape} → 读作: {X.shape[0]} 行 × {X.shape[1]} 列")
print(f"  ndim = {X.ndim} (2维)")

# 每一行是一个样本向量
print(f"\n第 0 个样本 (X[0]): shape={X[0].shape}, 前 5 维={X[0, :5].round(3)}")
print(f"第 15 个样本 (X[15]): shape={X[15].shape}")

# 每一列是一个特征（所有样本在该特征上的值）
print(f"\n第 0 个特征 (X[:, 0]): shape={X[:, 0].shape}, "
      f"均值={X[:, 0].mean():.3f}")

# ===== 矩阵索引：逗号左边选行，右边选列 =====
M = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])
print(f"\n矩阵 M (3×3):\n{M}")
print(f"M[0, 0] = {M[0, 0]}   ← 第 0 行第 0 列")
print(f"M[1, :] = {M[1, :]}    ← 第 1 行全部列 (得到一个向量)")
print(f"M[:, 2] = {M[:, 2]}    ← 全部行第 2 列 (得到一个向量)")
print(f"M[:2, 1:] = ...\n{M[:2, 1:]}  ← 前2行 × 后2列 (得到一个子矩阵)")

# ===== 全连接层的参数矩阵 =====
d_in, d_out = 784, 256
W = np.random.randn(d_in, d_out) * 0.01  # 权重矩阵 784×256
b = np.zeros(d_out)                       # 偏置向量 (256,)
print(f"\n全连接层参数: W.shape={W.shape}, b.shape={b.shape}")
print(f"规则: 输入维度({d_in}) @ 权重矩阵({d_in}×{d_out}) + 偏置({d_out}) → 输出({d_out})")




> **关键洞察**：矩阵 = "向量组成的表格"。第一维是"样本编号"，第二维是"特征编号"。这个约定不是任意的——PyTorch 的 `nn.Linear(784, 256)` 期望输入 shape 为 `(batch, 784)`，输出 `(batch, 256)`。如果你把 shape 写反了（`(784, batch)`），代码不会报语法错误，但计算结果会完全错误——这是 AI 工程中最常见的"静默 bug"之一。

🔗 **AI 连接**：矩阵乘法（第 6 章）是神经网络 99% 计算量的来源——`X @ W` 这个操作就是一层全连接网络的前向传播。第 29 章 Transformer 中的 Q、K、V 投影（`X @ W_Q`）也都是矩阵乘法。你此刻建立的"矩阵 = 批量向量"的直觉，会在第 6 章直接转化为"线性变换 = 空间扭曲"的几何理解。

---

### 3.4　张量（Tensor）—— 3 维及以上，AI 真正的"数据母体"

如果你同时处理 32 张 RGB 彩色图片，每张 224×224 像素、3 个颜色通道。数据形状是 `(32, 224, 224, 3)`——四个维度，远超矩阵的二维。这就是**张量**：矩阵的维度升级版。

📐 **定义　张量（Tensor）**：3 维及以上的多维数组。在深度学习中，"张量"也常泛指任意维度的数组（包括标量、向量、矩阵）。PyTorch 的核心数据类型就叫 `torch.Tensor`——它统一了 0 维到 N 维的所有数组。

张量最重要的能力不是"维度多"，而是**用 shape 一眼读懂数据的物理含义**。这是 AI 工程师的日常基本功：

| 数据 | shape | 读法 |
|------|-------|------|
| 单张灰度图 (MNIST) | `(28, 28)` | 28 行 × 28 列 像素 |
| 单张 RGB 图 | `(224, 224, 3)` | 高 224 × 宽 224 × 3 通道 |
| 批量灰度图 | `(32, 28, 28)` | 32 张 × 28 × 28 |
| 批量 RGB 图 | `(32, 224, 224, 3)` | 32 张 × 高 × 宽 × 通道 |
| Transformer 输入 | `(batch, seq_len, d_model)` | 批大小 × 序列长度 × 嵌入维度 |
| 多头注意力 Q | `(batch, n_heads, seq_len, d_k)` | 批 × 头数 × 序列长 × 每头维度 |

💻 **代码　亲手操作 3D 和 4D 张量**




In [ ]:
import numpy as np

# ===== 3D 张量：批量灰度图 =====
batch, height, width = 32, 28, 28
images = np.random.randn(batch, height, width)  # 模拟 MNIST 批量
print(f"批量灰度图: shape={images.shape}")
print(f"  axis=0 (batch): {images.shape[0]} 张图片")
print(f"  axis=1 (height): {images.shape[1]} 像素高")
print(f"  axis=2 (width): {images.shape[2]} 像素宽")
print(f"  第 0 张图片: shape={images[0].shape} ← 取出一个 28×28 矩阵")

# ===== 4D 张量：批量 RGB 图 =====
images_rgb = np.random.randn(32, 224, 224, 3)  # 32 张 224×224 RGB
print(f"\n批量 RGB 图: shape={images_rgb.shape}")
print(f"  第 0 张图片, 第 0 个通道: shape={images_rgb[0, :, :, 0].shape}")

# ===== Transformer 风格的 3D 张量 =====
seq_len, d_model = 10, 512
X_transformer = np.random.randn(batch, seq_len, d_model)
print(f"\nTransformer 输入: shape={X_transformer.shape}")
print(f"  第 0 个样本, 第 3 个 token: shape={X_transformer[0, 3].shape}")
print(f"  — 即一个 {d_model} 维的词嵌入向量")

# ===== 张量的 ndim 和 shape 是理解代码的第一入口 =====
print(f"\n{'数据':<25} {'shape':<25} {'ndim':<6}")
print("-" * 56)
for name, arr in [("标量(loss)", np.array(2.37)),
                   ("向量(用户特征)", np.array([28, 15000, 0.7])),
                   ("矩阵(批量输入)", np.random.randn(32, 784)),
                   ("3D张量(灰度图批量)", images),
                   ("4D张量(RGB图批量)", images_rgb)]:
    print(f"{name:<25} {str(arr.shape):<25} {arr.ndim:<6}")




> **关键洞察**：shape 是 AI 代码的"共同语言"。当你看到一个 bug 信息说 `RuntimeError: shape '[32, 512]' is invalid for input of size 131072`，你不需要看代码就能心算：32×512=16384 ≠ 131072，说明某个地方的维度设错了。**"读 shape 如读母语"是 AI 工程师的核心竞争力。**

🔗 **AI 连接**：Transformer 的核心张量流：`(batch, seq_len, d_model)` → 多头重塑为 `(batch, n_heads, seq_len, d_k)` → Q@Kᵀ 得 `(batch, n_heads, seq_len, seq_len)` → softmax 后的注意力权重 @ V 得 `(batch, n_heads, seq_len, d_v)`。这四步 shape 变换就是第 29 章的精华——如果你此刻就能自信地追踪 shape 变化，第 29 章读起来会像看小说一样流畅。

---

### 3.5　轴（Axis）—— `axis=0` 到底往哪边压？

这是本章最容易卡住、也最重要的一节。

`np.sum(arr, axis=0)` 是什么意思？很多教程说"axis=0 按行求和，axis=1 按列求和"——这句话只对矩阵有效，对 3D 张量就失效了。正确的直觉是：**`axis=k` 意味着沿着第 k 个维度"压扁"，那个维度会消失。**

用大白话说：`sum(axis=0)` = "把所有样本摞在一起求和"（batch 维度消失），`sum(axis=1)` = "把每个样本的所有特征加起来"（特征维度消失）。

📐 **定义　轴（Axis）**：张量的第 k 个维度，索引从 0 开始。对 shape=(a, b, c) 的张量求和：`axis=0` 消去 a（结果 shape=(b, c)），`axis=1` 消去 b（结果 shape=(a, c)），`axis=2` 消去 c（结果 shape=(a, b)）。`axis=-1` 永远是最后一维。

💻 **代码　亲手验证：对 (2,3,4) 张量沿各轴求和，观察 shape 变化**




In [ ]:
import numpy as np

np.random.seed(42)
tensor = np.random.randn(2, 3, 4)  # shape=(2,3,4)
# 想象成：2 个样本 × 3 行 × 4 列

print(f"原始张量 shape: {tensor.shape}\n")

# axis=0: 消去第 0 维→把所有样本叠在一起求和
sum0 = tensor.sum(axis=0)
print(f"sum(axis=0): {tensor.shape} → {sum0.shape}  "
      f"(2个样本合并为1个，后面的3×4保留)")

# axis=1: 消去第 1 维→每样本内部的行求和
sum1 = tensor.sum(axis=1)
print(f"sum(axis=1): {tensor.shape} → {sum1.shape}  "
      f"(每样本的3行合并，剩下2个样本×4列)")

# axis=2 (即 axis=-1): 消去最后一维→每样本每行的列求和
sum2 = tensor.sum(axis=2)
sum_neg1 = tensor.sum(axis=-1)
print(f"sum(axis=2): {tensor.shape} → {sum2.shape}  "
      f"(每行4列合并为1个数，剩下2个样本×3行)")
print(f"sum(axis=-1) 结果相同: {(sum2 == sum_neg1).all()} ✓")

# ===== 可视化：用一个小矩阵直观理解 axis =====
M = np.array([[1, 2, 3],
              [4, 5, 6]])
print(f"\n矩阵 M (2×3):\n{M}")
print(f"sum(axis=0) = {M.sum(axis=0)}  ← 竖向压扁: 每列求和, shape={M.sum(axis=0).shape}")
print(f"sum(axis=1) = {M.sum(axis=1)}  ← 横向压扁: 每行求和, shape={M.sum(axis=1).shape}")

# ===== 口诀 =====
print(f"\n口诀：axis=k 就是'沿着第k维的方向把数字加在一起'"
      f"——加完之后第k维消失")
print(f"对矩阵: axis=0 是'从上往下压'(行消失), "
      f"axis=1 是'从左往右压'(列消失)")




> **关键洞察**：axis 的直觉可以浓缩为一句话——**axis 的值就是"要被消灭的维度编号"**。这和 PyTorch 的 `softmax(dim=-1)`、`torch.cat(tensors, dim=0)`、`torch.stack(tensors, dim=1)` 共用完全相同的逻辑。一旦你用 `sum` 把 axis 直觉建立起来，所有其他沿轴操作都是一通百通。

🔗 **AI 连接**：第 22 章 softmax 的 `dim=-1` 就是沿最后一维做归一化（把每个 token 的 50000 个 logit 变成概率和为 1）；第 29 章 attention 中 `softmax(dim=-1)` 对每一行（每个 query 对所有 key 的得分）做归一化；第 21 章 LayerNorm 沿 `dim=-1` 对每个 token 的嵌入维度内部做归一化。这三个操作背后的 axis 直觉，全部来自你此刻建立的"axis = 要被消灭的维度"这条规则。

---



**✏️ 习题**

> 在下方新建代码单元格作答。
1. （概念）张量的 shape 为 `(32, 10, 512)`。请用一句中文描述这代表什么样的数据？其中 32、10、512 分别代表什么？
2. （概念）对 shape=(4, 3, 2) 的张量做 `sum(axis=1)`，结果的 shape 是什么？为什么？
3. （概念）`(768,)`、`(768, 1)`、`(1, 768)` 三个 shape 有什么区别？为什么 AI 代码中要区分它们？
4. （代码）创建一个 shape=(2, 3, 4) 的张量，分别对 axis=0、axis=1、axis=2 求和，打印每次求和前后的 shape 变化。再对同一张量做 `mean(axis=-1)`，验证结果的 shape。
5. （代码）模拟一个推荐系统场景：10 个用户，每个用户由 5 个特征描述（年龄、收入、活跃度、点击次数、购买次数）。创建一个 shape=(10, 5) 的矩阵，计算：
   - 所有用户的平均特征向量（shape 应该是什么？）
   - 每个用户特征的全局均值（shape 应该是什么？）
---
> 预览：下一章——**向量的加减与数乘**，AI 世界里的"平移"与"缩放"。
